# StART — Model-Risk Review (interactive)

Data-first review with every stage visible: discovery → target → task →
split → feature engineering → recommendation → execution → metrics →
explainability → sensitivity → robustness → evidence → agentic
challenge/governance/sign-off → AI-engineering stages → report.

Select the kernel **Python (StART .venv-start)**. Deterministic mode is the
default and needs no key; the LLM (if selected) reasons only over the
evidence bundle, never raw data.


## 1. Options
Edit directly, or use the widget cell below if `ipywidgets` is installed.


In [ ]:
OPTIONS = {
    'dataset_path': '',          # blank = built-in demo dataset
    'target_column': 'attrition',
    'split_strategy': 'stratified',
    'architecture': 'mlp',       # mlp | residual_mlp | wide_deep
    'activation': 'relu',        # relu | leaky_relu | gelu | tanh | selu | elu
    'agent_mode': 'deterministic',  # deterministic | llm
    'llm_provider': 'none',      # none | openai | anthropic | grok | enterprise_llm_gateway
    'run_dl': True,
}
print(OPTIONS)

## 2. Optional interactive widgets


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
    w_target = widgets.Text(value=OPTIONS['target_column'], description='target')
    w_split = widgets.Dropdown(options=['random','stratified','time_based','group','custom'], value=OPTIONS['split_strategy'], description='split')
    w_arch = widgets.Dropdown(options=['mlp','residual_mlp','wide_deep'], value=OPTIONS['architecture'], description='arch')
    w_act = widgets.Dropdown(options=['relu','leaky_relu','gelu','tanh','selu','elu'], value=OPTIONS['activation'], description='activation')
    w_mode = widgets.Dropdown(options=['deterministic','llm'], value=OPTIONS['agent_mode'], description='agent_mode')
    w_prov = widgets.Dropdown(options=['none','openai','anthropic','grok','enterprise_llm_gateway'], value=OPTIONS['llm_provider'], description='provider')
    w_dl = widgets.Checkbox(value=OPTIONS['run_dl'], description='run_dl')
    display(w_target, w_split, w_arch, w_act, w_mode, w_prov, w_dl)
    _W = (w_target, w_split, w_arch, w_act, w_mode, w_prov, w_dl)
except ImportError:
    _W = None
    print('ipywidgets not installed; using OPTIONS above.')

## 3. Load data


In [ ]:
if _W is not None:
    wt, ws, wa, wac, wm, wp, wd = _W
    OPTIONS.update(target_column=wt.value, split_strategy=ws.value, architecture=wa.value,
                   activation=wac.value, agent_mode=wm.value, llm_provider=wp.value, run_dl=wd.value)

path = OPTIONS['dataset_path'].strip()
if path:
    from start.data.loaders import load_any_tabular
    df = load_any_tabular(path)
    target = OPTIONS['target_column'] or None
else:
    from start.modeling.data import load_attrition_dataset
    df = load_attrition_dataset(seed=42)
    target = OPTIONS['target_column'] or 'attrition'
print(f'{len(df)} rows x {df.shape[1]} columns | target: {target}')

## 4. Resolve LLM (deterministic by default; LLM never sees raw data)


In [ ]:
llm = None
if OPTIONS['agent_mode'] == 'llm' and OPTIONS['llm_provider'] not in ('none',''):
    from start.core.config import LLMConfig
    from start.providers.llm import get_llm_provider
    llm = get_llm_provider(LLMConfig(provider=OPTIONS['llm_provider']))
    print('LLM provider:', OPTIONS['llm_provider'], '| available:', getattr(llm,'available',False))
else:
    print('Deterministic mode — no key required.')

## 5. Run the full review (each stage prints as it runs)


In [ ]:
from start.modeling.review_orchestrator import ReviewOrchestrator

def show(e):
    mark = {'running':'·','complete':'✓','skipped':'—'}.get(e.status,' ')
    detail = f'  {e.detail}' if e.detail else ''
    print(f"  {mark} {e.stage.replace('_',' ').title():28s} [{e.status}]{detail}")

orch = ReviewOrchestrator(on_stage=show)
outcome = orch.run(df, user_target=target, split_strategy=OPTIONS['split_strategy'],
                   agent_mode=OPTIONS['agent_mode'], llm=llm, output_root='start_output',
                   run_dl=OPTIONS['run_dl'], seed=42)

## 6. Summary, evidence, and AI-engineering surface


In [ ]:
import pandas as pd
print('run:', outcome.run_id, '| task:', outcome.task_type, '| modality:', outcome.modality)
print('recommended family:', outcome.recommended_family)
print('evidence critique:', 'PASSED' if outcome.agent_review.critique_ok else 'FAILED')
display(pd.DataFrame([{'test_id': r.test_id, 'name': r.test_name, 'status': r.status.value} for r in outcome.evidence]))
if outcome.cohort_metrics:
    display(pd.DataFrame(outcome.cohort_metrics).T)
print('\nsign-off:', outcome.agent_review.signoff)
print('report:', outcome.report_path)
display(pd.DataFrame([{'stage': s.name, 'category': s.category, 'status': s.status} for s in outcome.ai_engineering]))